In [7]:
import time
import os
import re
import urllib.parse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import logging

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def download_laws_by_keyword(keyword, max_pages=10):
    """
    키워드로 검색된 모든 법령 다운로드
    
    Args:
        keyword (str): 검색할 키워드 (예: 금융, 온라인, 은행, 보험 등)
        max_pages (int): 최대 처리할 페이지 수 (기본값: 10)
    """
    
    # 다운로드 폴더 설정
    safe_keyword = re.sub(r'[<>:"/\\|?*]', '_', keyword)  # 폴더명에 사용할 수 없는 문자 제거
    download_path = os.path.abspath(f"./{safe_keyword}_법령_다운로드")
    if not os.path.exists(download_path):
        os.makedirs(download_path)
    
    # Chrome 옵션 설정
    chrome_options = Options()
    
    # 다운로드 설정
    prefs = {
        "download.default_directory": download_path,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    
    # 봇 탐지 우회 설정
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = None
    total_success = 0
    total_failed = 0
    
    try:
        # Chrome 드라이버 시작
        driver = webdriver.Chrome(options=chrome_options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        wait = WebDriverWait(driver, 20)
        
        logger.info(f"=== '{keyword}' 키워드 법령 다운로드 시작 ===")
        
        # 키워드로 법령 검색 페이지 이동
        encoded_keyword = urllib.parse.quote(keyword, encoding='utf-8')
        search_url = f"https://www.law.go.kr/lsSc.do?section=&menuId=1&subMenuId=15&tabMenuId=81&eventGubun=060101&query={encoded_keyword}"
        logger.info(f"검색 키워드: {keyword}")
        logger.info(f"검색 URL: {search_url}")
        
        driver.get(search_url)
        time.sleep(5)
        
        # 전체 법령 개수 확인
        try:
            # 검색 결과 개수 확인 (예: "총 58건")
            result_count_element = driver.find_element(By.XPATH, "//span[contains(text(), '총') and contains(text(), '건')]")
            result_count_text = result_count_element.text
            total_count = int(re.search(r'총\s*(\d+)\s*건', result_count_text).group(1))
            logger.info(f"'{keyword}' 키워드로 총 {total_count}개의 법령이 검색되었습니다.")
        except:
            logger.warning("검색 결과 개수를 확인할 수 없습니다. 계속 진행...")
            total_count = 0
        
        # 페이지별 처리
        current_page = 1
        processed_laws = 0
        
        while current_page <= max_pages:
            logger.info(f"\n=== {current_page}페이지 처리 시작 ===")
            
            # 2페이지부터는 페이지 이동
            if current_page > 1:
                try:
                    # 해당 페이지 번호 링크 찾기
                    page_link = driver.find_element(By.XPATH, f"//a[contains(@onclick, \"movePage('{current_page}')\")]")
                    driver.execute_script("arguments[0].click();", page_link)
                    logger.info(f"{current_page}페이지로 이동 성공")
                    time.sleep(3)
                except:
                    # 페이지 링크가 없으면 종료
                    logger.info(f"{current_page}페이지 링크가 없습니다. 다운로드 완료.")
                    break
            
            # 현재 페이지의 표시된 법령 링크 찾기
            try:
                all_law_links = driver.find_elements(By.XPATH, "//a[contains(@onclick, 'lsViewWideAll') and not(contains(@style, 'display: none'))]")
                
                # 실제로 화면에 보이는 링크만 필터링
                visible_links = []
                for link in all_law_links:
                    try:
                        if link.is_displayed():
                            visible_links.append(link)
                    except:
                        continue
                
                all_law_links = visible_links
                logger.info(f"{current_page}페이지에서 {len(all_law_links)}개 법령 발견")
                
                if len(all_law_links) == 0:
                    logger.info(f"{current_page}페이지에 법령이 없습니다. 다운로드 완료.")
                    break
                    
            except Exception as e:
                logger.error(f"{current_page}페이지에서 법령 링크를 찾을 수 없습니다: {e}")
                break
            
            # 현재 페이지의 각 법령 다운로드
            page_success = 0
            page_failed = 0
            
            for i, law_link in enumerate(all_law_links, 1):
                try:
                    processed_laws += 1
                    
                    logger.info(f"[{processed_laws}번째] {current_page}페이지 {i}번째 법령 처리 중...")
                    
                    # 법령명 확인
                    law_name = law_link.text.strip()
                    onclick = law_link.get_attribute('onclick')
                    logger.info(f"법령명: {law_name}")
                    
                    # 법령 클릭 (본문 표시)
                    try:
                        driver.execute_script("arguments[0].click();", law_link)
                    except:
                        # 대안: JavaScript 함수 직접 호출
                        params = re.findall(r"'([^']+)'", onclick)
                        if len(params) >= 8:
                            js_code = f"lsViewWideAll('{params[0]}','{params[1]}','{params[2]}',arguments[0],'{params[3]}','{params[4]}','{params[5]}','{params[6]}');"
                            driver.execute_script(js_code, law_link)
                        else:
                            page_failed += 1
                            continue
                    
                    # 본문 로딩 대기
                    time.sleep(4)
                    
                    # 서버 과부하 페이지 확인
                    if "사용자가 많아 요청하신 페이지를 정상적으로 제공할 수 없습니다" in driver.page_source:
                        logger.warning("서버 과부하 감지. 30초 대기 후 재시도...")
                        time.sleep(30)
                        
                        # 다시 법령 클릭 시도
                        try:
                            driver.execute_script("arguments[0].click();", law_link)
                            time.sleep(5)
                        except:
                            logger.error(f"재시도 실패: {law_name}")
                            page_failed += 1
                            continue
                    
                    # 본문 저장 버튼 클릭
                    try:
                        save_btn = wait.until(EC.element_to_be_clickable((By.ID, "bdySaveBtn")))
                        driver.execute_script("arguments[0].click();", save_btn)
                        time.sleep(3)
                    except Exception as e:
                        logger.error(f"저장 버튼 클릭 실패: {law_name} - {e}")
                        page_failed += 1
                        continue
                    
                    # DOC 옵션 선택
                    try:
                        doc_radio = wait.until(EC.element_to_be_clickable((By.ID, "FileSaveDoc1")))
                        driver.execute_script("arguments[0].click();", doc_radio)
                        time.sleep(1)
                    except:
                        pass  # DOC 옵션이 없으면 기본 옵션으로 진행
                    
                    # 팝업 저장 버튼 클릭
                    try:
                        popup_save_btn = wait.until(EC.element_to_be_clickable((By.ID, "aBtnOutPutSave")))
                        driver.execute_script("arguments[0].click();", popup_save_btn)
                        time.sleep(3)
                    except Exception as e:
                        logger.error(f"팝업 저장 실패: {law_name} - {e}")
                        page_failed += 1
                        continue
                    
                    logger.info(f"다운로드 완료: {law_name}")
                    page_success += 1
                    
                    # 다음 법령을 위한 대기 (랜덤)
                    import random
                    wait_time = random.uniform(3, 7)
                    logger.info(f"서버 부하 방지를 위해 {wait_time:.1f}초 대기...")
                    time.sleep(wait_time)
                    
                except Exception as e:
                    logger.error(f"법령 다운로드 실패: {law_name} - {e}")
                    page_failed += 1
                    continue
            
            # 페이지 결과
            logger.info(f"{current_page}페이지 완료: 성공 {page_success}개, 실패 {page_failed}개")
            total_success += page_success
            total_failed += page_failed
            
            # 다음 페이지로
            current_page += 1
        
        # 최종 결과
        logger.info(f"\n=== '{keyword}' 키워드 다운로드 완료 ===")
        logger.info(f"처리된 페이지: {current_page - 1}개")
        logger.info(f"성공: {total_success}개")
        logger.info(f"실패: {total_failed}개")
        logger.info(f"전체: {total_success + total_failed}개")
        
        # 다운로드 파일 확인
        downloaded_files = [f for f in os.listdir(download_path) if f.endswith(('.doc', '.docx', '.hwp', '.pdf'))]
        logger.info(f"다운로드 폴더 내 파일 개수: {len(downloaded_files)}")
        
        print(f"\n'{keyword}' 키워드의 모든 법령이 '{download_path}' 폴더에 다운로드되었습니다.")
        
    except Exception as e:
        logger.error(f"전체 프로세스 실패: {e}")
        
    finally:
        if driver:
            driver.quit()
            logger.info("브라우저 종료")

def main():
    """메인 실행 함수"""
    print("=== 법령 키워드 다운로더 ===")
    print("예시 키워드: 금융, 온라인, 은행, 보험, 증권, 핀테크, 블록체인, 암호화폐 등")
    
    # 키워드 입력받기
    keyword = input("\n다운로드할 법령의 키워드를 입력하세요: ").strip()
    
    if not keyword:
        print("키워드를 입력해주세요.")
        return
    
    # 최대 페이지 수 입력받기
    try:
        max_pages_input = input("최대 처리할 페이지 수를 입력하세요 (기본값: 10): ").strip()
        max_pages = int(max_pages_input) if max_pages_input else 10
    except:
        max_pages = 10
    
    print(f"\n'{keyword}' 키워드로 최대 {max_pages}페이지까지 법령을 다운로드합니다...")
    
    # 다운로드 실행
    download_laws_by_keyword(keyword, max_pages)

if __name__ == "__main__":
    main()

=== 법령 키워드 다운로더 ===
예시 키워드: 금융, 온라인, 은행, 보험, 증권, 핀테크, 블록체인, 암호화폐 등

다운로드할 법령의 키워드를 입력하세요: 상법
최대 처리할 페이지 수를 입력하세요 (기본값: 10): 1

'상법' 키워드로 최대 1페이지까지 법령을 다운로드합니다...


2025-08-07 15:09:10,836 - INFO - === '상법' 키워드 법령 다운로드 시작 ===
2025-08-07 15:09:10,837 - INFO - 검색 키워드: 상법
2025-08-07 15:09:10,838 - INFO - 검색 URL: https://www.law.go.kr/lsSc.do?section=&menuId=1&subMenuId=15&tabMenuId=81&eventGubun=060101&query=%EC%83%81%EB%B2%95
2025-08-07 15:09:18,753 - WARNING - 검색 결과 개수를 확인할 수 없습니다. 계속 진행...
2025-08-07 15:09:18,753 - INFO - 
=== 1페이지 처리 시작 ===
2025-08-07 15:09:18,867 - INFO - 1페이지에서 4개 법령 발견
2025-08-07 15:09:18,867 - INFO - [1번째] 1페이지 1번째 법령 처리 중...
2025-08-07 15:09:18,894 - INFO - 법령명: 1.  상법
[시행 2027. 1. 1.] [법률 제20991호, 2025. 7. 22., 일부개정]
2025-08-07 15:09:35,704 - INFO - 다운로드 완료: 1.  상법
[시행 2027. 1. 1.] [법률 제20991호, 2025. 7. 22., 일부개정]
2025-08-07 15:09:35,704 - INFO - 서버 부하 방지를 위해 6.7초 대기...
2025-08-07 15:09:42,414 - INFO - [2번째] 1페이지 2번째 법령 처리 중...
2025-08-07 15:09:42,456 - INFO - 법령명: 2.  상법
[시행 2026. 7. 23.] [법률 제20991호, 2025. 7. 22., 일부개정]
2025-08-07 15:10:02,222 - INFO - 다운로드 완료: 2.  상법
[시행 2026. 7. 23.] [법률 제20991호, 2025. 7. 22., 일부개정]
202


'상법' 키워드의 모든 법령이 'C:\ai_x\source\Project2nd\금융법령수집\상법_법령_다운로드' 폴더에 다운로드되었습니다.


2025-08-07 15:10:53,139 - INFO - 브라우저 종료
